In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import sys
sys.path.append("..")
from src.preprocessing import df_to_densities
from src.forecasting import cv


import warnings
from scipy.integrate import IntegrationWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=IntegrationWarning)

Steps: <br>
1. Use cross-validation to select the best parameters for in-sample KDE <br>
2. Use cross-validation to select the number of dimensions for the dFPC using the resulting parameters for KDE in 1.

In [ ]:
# Data
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df = pd.read_excel(returns_path, index_col="time")

# 1. Selecting KDE parameters

In [ ]:
# Parameters to cross-validate
density_param_grid_0 = [
    {'kernel': ['gaussian', 'epanechnikov'],
        'bandwidth': ['silverman', 'scott'],#, 'cv'],
        'adaptive': [True, False]}, 
    {'kernel': ['t_student'], 
        'df': range(2,6),
        'bandwidth': ['silverman', 'scott'],#, 'cv'],
        'adaptive': [True, False]}]

density_param_grid = list(ParameterGrid(density_param_grid_0))
len(density_param_grid)

In [ ]:
params_dict = {}
records = []

normalize_options = [True, False]
total_params = len(density_param_grid) * len(normalize_options)
i=1
for normalize in normalize_options:
    for params in density_param_grid:
        print(f"({i}/{total_params}) | Normalize = {normalize} | {params}")
        i += 1

        df_support, df_densities = df_to_densities(
                                        df, 
                                        params, 
                                        normalize_densities=normalize,
                                        verbose=False)
        try:
            cv_measures = cv(df_densities, df_support, initial_window=100)
        except Exception as e:
            print(f"\t ERROR: {e}")
            continue
        for m in cv_measures:
            record = {
                # density
                "normalized_density": normalize,
                
                # parameters
                "kernel": params["kernel"],
                "bandwidth": params["bandwidth"],
                "adaptive": params["adaptive"],

                # CV info
                "fold": m["fold"]+1,
                "method": m["method"],

                # metrics
                "KLD": float(m["KLD"]),
                "JSD": float(m["JSD"]),
                "L_1": float(m["L_1"]),
                "L_2": float(m["L_2"]),
                "L_INFTY": float(m["L_INFTY"]),
            }

            records.append(record)
            
results_df = pd.DataFrame(records)

In [ ]:
results_df.to_excel("../data/processed/cv_density_estimation_v2.xlsx", index=False)

In [ ]:
metrics = ["KLD", "JSD", "L_1", "L_2", "L_INFTY"]

mean_df = (
    results_df
    .groupby(["density_model", "kernel", "bandwidth", "adaptive"])[metrics]
    .mean()
    .reset_index()
)
for m in metrics:
    best_value = mean_df[m].min()
    mean_df[f"win_{m}"] = mean_df[m] == best_value

win_cols = [f"win_{m}" for m in metrics]

mean_df["n_wins"] = mean_df[win_cols].sum(axis=1)

best_overall = mean_df.sort_values("n_wins", ascending=False).iloc[0]
best_overall